# SentinelID - Phase 1: OCR + MRZ Extraction (P1)
Run each cell in order. Upload a sample passport/ID image when prompted.

In [ ]:
!apt-get install tesseract-ocr -y -qq
!pip install pytesseract easyocr passporteye opencv-python-headless pillow -q

## Step 1: Upload a test image

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload a sample passport/ID image
image_path = list(uploaded.keys())[0]

## Step 2: Basic OCR pass

In [ ]:
import pytesseract
from PIL import Image

def extract_text(image_path):
    img = Image.open(image_path)
    return pytesseract.image_to_string(img)

print(extract_text(image_path))

## Step 3: MRZ-specific parsing
The two lines of machine-readable text at the bottom of passports - most reliable structured field source.

In [ ]:
from passporteye import read_mrz

def extract_mrz(image_path):
    mrz = read_mrz(image_path)
    if mrz is None:
        return None
    return mrz.to_dict()

mrz_data = extract_mrz(image_path)
mrz_data

## Step 4: Preprocessing for bad scans

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

def preprocess(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    cv2.imwrite("preprocessed.jpg", thresh)
    return "preprocessed.jpg"

pre_path = preprocess(image_path)
cv2_imshow(cv2.imread(pre_path))

## Step 5: Full pipeline function
This is the function to export to your teammates for Phase 2 backend wiring.

In [ ]:
def run_ocr_pipeline(image_path):
    mrz_data = extract_mrz(image_path)
    raw_text = extract_text(image_path)
    return {"mrz": mrz_data, "raw_text": raw_text}

import json
result = run_ocr_pipeline(image_path)
print(json.dumps(result, indent=2, default=str))

## Exit test
Output above should show name, passport number, DOB, expiry, and nationality correctly for at least 3 of 5 uploaded samples.

Download this notebook as .py via File > Download > Download .py once done.